# Blood Cell Classification - Complete End-to-End Notebook

This notebook does the full project workflow by itself:
- crop the raw YOLO annotations into cell images
- split them into train/val/test
- train the model
- evaluate the model on held-out test data
- compare multiple architectures
- save the final results

This is a self-contained notebook version of the project, so you do not need to depend on separate scripts to get the main results.

## 1. Setup and configuration

Load the project config, select the device, and define the fixed seed for reproducibility.

In [ ]:
from pathlib import Path
import json
import random
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

project_root = Path.cwd()
config_path = project_root / 'configs' / 'config.yaml'
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Project root:', project_root)
print('Device:', DEVICE)
print('Classes:', cfg['classes'])
print('Models:', cfg['models'])

## 2. Crop the raw dataset into processed cell images

This step reads the YOLO label files and crops each cell region into the folder structure required for training.

In [ ]:
def read_split_file(path_str):
    p = Path(path_str)
    if not p.exists():
        return set()
    with open(p, 'r') as f:
        return {Path(line.strip()).stem for line in f if line.strip()}


def assign_splits(image_stems, cfg):
    train_set = read_split_file(cfg['split_files']['train'])
    val_set = read_split_file(cfg['split_files']['val'])
    test_set = read_split_file(cfg['split_files']['test'])

    if train_set or val_set or test_set:
        assignment = {}
        for stem in image_stems:
            if stem in train_set:
                assignment[stem] = 'train'
            elif stem in val_set:
                assignment[stem] = 'val'
            elif stem in test_set:
                assignment[stem] = 'test'
            else:
                assignment[stem] = 'train'
        return assignment

    shuffled = image_stems.copy()
    random.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    assignment = {}
    for i, stem in enumerate(shuffled):
        if i < n_train:
            assignment[stem] = 'train'
        elif i < n_train + n_val:
            assignment[stem] = 'val'
        else:
            assignment[stem] = 'test'
    return assignment


def yolo_box_to_pixels(x_c, y_c, w, h, img_w, img_h):
    x1 = int((x_c - w / 2) * img_w)
    y1 = int((y_c - h / 2) * img_h)
    x2 = int((x_c + w / 2) * img_w)
    y2 = int((y_c + h / 2) * img_h)
    return max(0, x1), max(0, y1), min(img_w, x2), min(img_h, y2)


raw_images_dir = project_root / cfg['raw_images_dir']
raw_labels_dir = project_root / cfg['raw_labels_dir']
processed_dir = project_root / cfg['processed_dir']
classes = cfg['classes']
image_size = cfg['image_size']

label_files = sorted(raw_labels_dir.glob('*.txt'))
if not label_files:
    raise FileNotFoundError(f'No TXT labels found in {raw_labels_dir}')

image_stems = [p.stem for p in label_files]
split_assignment = assign_splits(image_stems, cfg)

for split in ['train', 'val', 'test']:
    for cls in classes:
        (processed_dir / split / cls).mkdir(parents=True, exist_ok=True)

counts = {split: {cls: 0 for cls in classes} for split in ['train', 'val', 'test']}
skipped = 0

for label_file in label_files:
    stem = label_file.stem
    split = split_assignment.get(stem, 'train')

    img_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        candidate = raw_images_dir / f'{stem}{ext}'
        if candidate.exists():
            img_path = candidate
            break
    if img_path is None:
        skipped += 1
        continue

    image = Image.open(img_path).convert('RGB')
    img_w, img_h = image.size

    with open(label_file, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    for i, line in enumerate(lines):
        parts = line.split()
        if len(parts) != 5:
            continue
        cls_idx, x_c, y_c, w, h = parts
        cls_idx = int(cls_idx)
        if cls_idx >= len(classes):
            continue
        cls_name = classes[cls_idx]

        x1, y1, x2, y2 = yolo_box_to_pixels(float(x_c), float(y_c), float(w), float(h), img_w, img_h)
        if x2 <= x1 or y2 <= y1:
            continue

        crop = image.crop((x1, y1, x2, y2)).resize((image_size, image_size))
        out_path = processed_dir / split / cls_name / f'{stem}_{i}.png'
        crop.save(out_path)
        counts[split][cls_name] += 1

print('Crop counts:')
for split in ['train', 'val', 'test']:
    print(split, counts[split])
if skipped:
    print(f'Skipped {skipped} files due to missing source image.')

## 3. Inspect the generated dataset

Confirm that the processed folders exist and count the images in each class.

In [ ]:
for split in ['train', 'val', 'test']:
    print(f'\n[{split}]')
    for cls in classes:
        folder = processed_dir / split / cls
        count = len(list(folder.glob('*.png'))) if folder.exists() else 0
        print(f'  {cls}: {count}')

## 4. Build the dataset and samplers

Here we define the custom dataset, transforms, and weighted sampler for class imbalance.

In [ ]:
class CellCropDataset(Dataset):
    def __init__(self, root_dir, classes, transform=None):
        self.root_dir = Path(root_dir)
        self.classes = classes
        self.transform = transform
        self.samples = []

        for idx, cls_name in enumerate(classes):
            cls_dir = self.root_dir / cls_name
            if not cls_dir.exists():
                continue
            for img_path in sorted(cls_dir.glob('*.png')):
                self.samples.append((img_path, idx))

        if not self.samples:
            raise RuntimeError(f'No images found under {root_dir}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

    def class_counts(self):
        counts = [0] * len(self.classes)
        for _, label in self.samples:
            counts[label] += 1
        return counts


def build_transforms(cfg, train):
    mean = cfg['imagenet_mean']
    std = cfg['imagenet_std']
    size = cfg['image_size']
    if train:
        aug = cfg['augment']
        return transforms.Compose([
            transforms.Resize((size, size)),
            transforms.RandomRotation(aug['rotation_degrees']),
            transforms.RandomHorizontalFlip(p=aug['horizontal_flip_prob']),
            transforms.ColorJitter(
                brightness=aug['brightness_jitter'],
                contrast=aug['contrast_jitter'],
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ])
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])


def build_weighted_sampler(dataset):
    counts = dataset.class_counts()
    class_weights = [1.0 / c if c > 0 else 0.0 for c in counts]
    weights = [class_weights[label] for _, label in dataset.samples]
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)


def build_dataloaders(cfg):
    train_ds = CellCropDataset(processed_dir / 'train', classes, build_transforms(cfg, train=True))
    val_ds = CellCropDataset(processed_dir / 'val', classes, build_transforms(cfg, train=False))
    test_ds = CellCropDataset(processed_dir / 'test', classes, build_transforms(cfg, train=False))

    train_sampler = build_weighted_sampler(train_ds)
    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], sampler=train_sampler, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=0)

    return train_loader, val_loader, test_loader, train_ds.class_counts()


train_loader, val_loader, test_loader, class_counts = build_dataloaders(cfg)
print('Training class counts:', dict(zip(cfg['classes'], class_counts)))
print('Train batches:', len(train_loader))
print('Val batches:', len(val_loader))
print('Test batches:', len(test_loader))

## 5. Build the backbone models

Define the three transfer-learning backbones used in the project.

In [ ]:
def build_model(model_name, num_classes):
    if model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_classes))
    elif model_name == 'mobilenet_v3_small':
        model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, num_classes)
    elif model_name == 'densenet121':
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        in_features = model.classifier.in_features
        model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_classes))
    else:
        raise ValueError(f'Unknown model name: {model_name}')
    return model


print('Model builder ready.')

## 6. Train a model in this notebook

This cell trains one backbone directly inside the notebook using the same setup as the project.

In [ ]:
def compute_class_weights(class_counts, beta=None):
    counts = torch.tensor(class_counts, dtype=torch.float)
    if beta is None:
        weights = 1.0 / counts
    else:
        effective_num = 1.0 - torch.pow(torch.tensor(beta), counts)
        weights = (1.0 - beta) / effective_num
    weights = weights / weights.sum() * len(counts)
    return weights


def train_one_model(model_name, cfg, epochs=30):
    model = build_model(model_name, len(cfg['classes'])).to(DEVICE)
    train_loader, val_loader, _, class_counts = build_dataloaders(cfg)

    weights = compute_class_weights(class_counts, cfg.get('effective_number_beta')).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'])

    best_val_f1 = -1.0
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_total = 0.0
        all_labels, all_preds = [], []

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            train_loss_total += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())

        train_loss = train_loss_total / len(train_loader.dataset)
        train_report = classification_report(all_labels, all_preds, labels=list(range(len(cfg['classes']))), target_names=cfg['classes'], output_dict=True, zero_division=0)
        train_f1 = train_report['macro avg']['f1-score']

        model.eval()
        val_labels, val_preds = [], []
        val_loss_total = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)
                logits = model(images)
                loss = criterion(logits, labels)
                val_loss_total += loss.item() * images.size(0)
                preds = logits.argmax(dim=1)
                val_labels.extend(labels.cpu().tolist())
                val_preds.extend(preds.cpu().tolist())

        val_loss = val_loss_total / len(val_loader.dataset)
        val_report = classification_report(val_labels, val_preds, labels=list(range(len(cfg['classes']))), target_names=cfg['classes'], output_dict=True, zero_division=0)
        val_f1 = val_report['macro avg']['f1-score']

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_f1': train_f1,
            'val_loss': val_loss,
            'val_f1': val_f1,
        })

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | train_f1={train_f1:.4f} | val_loss={val_loss:.4f} | val_f1={val_f1:.4f}')

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, best_val_f1


selected_model = 'efficientnet_b0'
model, history, best_val_f1 = train_one_model(selected_model, cfg, epochs=cfg['num_epochs'])
print(f'\nBest validation macro-F1 for {selected_model}: {best_val_f1:.4f}')

# plot loss and F1 curves
epochs = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]
train_f1s = [h['train_f1'] for h in history]
val_f1s = [h['val_f1'] for h in history]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label='Train loss')
plt.plot(epochs, val_losses, label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss curves')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, train_f1s, label='Train macro-F1')
plt.plot(epochs, val_f1s, label='Val macro-F1')
plt.xlabel('Epoch')
plt.ylabel('Macro-F1')
plt.title('F1 curve')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Evaluate the trained model

Compute the final test-set metrics, confusion matrix, and macro-AUC.

In [ ]:
def evaluate_model(model, cfg):
    _, _, test_loader, _ = build_dataloaders(cfg)
    model.eval()

    all_labels, all_preds = [], []
    all_probs = []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            logits = model(images)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(dim=1).cpu().tolist()
            all_labels.extend(labels.tolist())
            all_preds.extend(preds)
            all_probs.append(probs)

    all_probs = np.concatenate(all_probs, axis=0) if all_probs else np.zeros((len(all_labels), len(cfg['classes'])) )
    truth = label_binarize(all_labels, classes=list(range(len(cfg['classes']))))
    try:
        macro_auc = roc_auc_score(truth, all_probs, average='macro', multi_class='ovr')
    except Exception:
        macro_auc = None

    report = classification_report(all_labels, all_preds, target_names=cfg['classes'], output_dict=True, zero_division=0)
    print(classification_report(all_labels, all_preds, target_names=cfg['classes'], zero_division=0))

    cm = confusion_matrix(all_labels, all_preds, normalize='true')
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, cmap='Blues')
    plt.xticks(range(len(cfg['classes'])), cfg['classes'])
    plt.yticks(range(len(cfg['classes'])), cfg['classes'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Normalised confusion matrix')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = cm[i, j]
            plt.text(j, i, f'{val:.2f}', ha='center', va='center', color='black' if val > 0.5 else 'white')
    plt.tight_layout()
    plt.show()

    summary = {
        'Macro-F1': report['macro avg']['f1-score'],
        'Macro-AUC': macro_auc,
        'Accuracy': report['accuracy'],
    }
    return summary, cm


summary, cm = evaluate_model(model, cfg)
print('\nTest summary:')
print(summary)

## 8. Compare all three backbones

Run the same training and evaluation block for each model to compare their performance on the same task.

In [ ]:
all_results = []
for model_name in cfg['models']:
    print(f'\n=== TRAINING {model_name.upper()} ===')
    trained_model, history, best_val_f1 = train_one_model(model_name, cfg, epochs=cfg['num_epochs'])
    model_summary, conf_matrix = evaluate_model(trained_model, cfg)
    all_results.append({
        'Model': model_name,
        'BestValMacroF1': best_val_f1,
        'Macro-F1': model_summary['Macro-F1'],
        'Macro-AUC': model_summary['Macro-AUC'],
        'Accuracy': model_summary['Accuracy'],
    })

results_df = pd.DataFrame(all_results)
print('\nFinal comparison table:')
print(results_df)

## 9. Save final outputs

This saves the notebook results so they can be referenced later for the report or presentation.

In [ ]:
results_dir = project_root / 'outputs' / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

with open(results_dir / 'notebook_model_summary.json', 'w') as f:
    json.dump(all_results, f, indent=2)

results_df.to_csv(results_dir / 'notebook_model_summary.csv', index=False)
print('Saved notebook results to:', results_dir)

## 10. Final interpretation

This notebook now contains the full workflow in one place: raw data is cropped, training is performed, evaluation is carried out, and final model comparison is generated within the same notebook. That is the standard way to make an ML project notebook complete and independent.

The final report can still be written separately, but the notebook itself is now a complete experiment record.

## 11. Conclusion

This notebook is now a complete project notebook. It covers the key steps from data preparation to training and evaluation and is structured like the full project workflow that your friends were showing in their notebooks.